Initialization

In [51]:
# Import the QICK drivers and auxiliary libraries
from qick import *
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [79]:
GEN_CH = 4
RO_CH = 0

In [53]:
# Load bitstream with custom overlay
soc = QickSoc()
# Since we're running locally on the QICK, we don't need a separate QickConfig object.
# If running remotely, you could generate a QickConfig from the QickSoc:
#     soccfg = QickConfig(soc.get_cfg())
# or save the config to file, and load it later:
#     with open("qick_config.json", "w") as f:
#         f.write(soc.dump_cfg())
#     soccfg = QickConfig("qick_config.json")
soccfg = soc
print(soccfg)

QICK running on ZCU111, software version 0.2.406

Firmware configuration (built Wed Aug 16 13:39:03 2023):

	Global clocks (MHz): tProc dispatcher timing 384.000, RF reference 204.800
	Groups of related clocks: [tProc clock, DAC tile 0], [DAC tile 1], [ADC tile 0]

	7 signal generator channels:
	0:	axis_signal_gen_v6 - fs=6144.000 Msps, fabric=384.000 MHz
		envelope memory: 65536 complex samples (10.667 us)
		32-bit DDS, range=6144.000 MHz
		DAC tile 0, blk 0 is DAC228_T0_CH0, or RF board DAC port 0
	1:	axis_signal_gen_v6 - fs=6144.000 Msps, fabric=384.000 MHz
		envelope memory: 65536 complex samples (10.667 us)
		32-bit DDS, range=6144.000 MHz
		DAC tile 0, blk 1 is DAC228_T0_CH1, or RF board DAC port 1
	2:	axis_signal_gen_v6 - fs=6144.000 Msps, fabric=384.000 MHz
		envelope memory: 65536 complex samples (10.667 us)
		32-bit DDS, range=6144.000 MHz
		DAC tile 0, blk 2 is DAC228_T0_CH2, or RF board DAC port 2
	3:	axis_signal_gen_v6 - fs=6144.000 Msps, fabric=384.000 MHz
		envelope memo

Continous Single Tone Sine / Cosine Wave

In [66]:
class SingleTone(AveragerProgram):
    def initialize(self):
        cfg=self.cfg   
        res_ch = cfg["res_ch"]

        # set the nyquist zone
        self.declare_gen(ch=cfg["res_ch"], nqz=1)
        
        # configure the readout lengths and downconversion frequencies (ensuring it is an available DAC frequency)
        for ch in cfg["ro_chs"]:
            self.declare_readout(ch=ch, length=self.cfg["readout_length"],
                                 freq=self.cfg["pulse_freq"], gen_ch=cfg["res_ch"])

        # convert frequency to DAC frequency (ensuring it is an available ADC frequency)
        freq = self.freq2reg(cfg["pulse_freq"],gen_ch=res_ch, ro_ch=cfg["ro_chs"][0])
        phase = self.deg2reg(cfg["res_phase"], gen_ch=res_ch)
        gain = cfg["pulse_gain"]
        self.default_pulse_registers(ch=res_ch, freq=freq, phase=phase, gain=gain)
                
        style=self.cfg["pulse_style"]

        if style in ["flat_top","arb"]:
            sigma = cfg["sigma"]
            self.add_gauss(ch=res_ch, name="measure", sigma=sigma, length=sigma*5)
            
        if style == "const":
            self.set_pulse_registers(ch=res_ch, style=style, length=cfg["length"], mode="periodic")
        else:
            pass
        
        self.synci(200)  # give processor some time to configure pulses
    
    def body(self):
        # fire the pulse
        # trigger all declared ADCs
        # pulse PMOD0_0 for a scope trigger
        # pause the tProc until readout is done
        # increment the time counter to give some time before the next measurement
        # (the syncdelay also lets the tProc get back ahead of the clock)
        self.measure(pulse_ch=self.cfg["res_ch"], 
                     adcs=self.ro_chs,
                     pins=[0], 
                     adc_trig_offset=self.cfg["adc_trig_offset"],
                     wait=True,
                     syncdelay=self.us2cycles(self.cfg["relax_delay"]))
        
        # equivalent to the following:
#         self.trigger(adcs=self.ro_chs,
#                      pins=[0], 
#                      adc_trig_offset=self.cfg["adc_trig_offset"])
#         self.pulse(ch=self.cfg["res_ch"])
#         self.wait_all()
#         self.sync_all(self.us2cycles(self.cfg["relax_delay"]))

In [87]:
config={"res_ch":GEN_CH, # --Fixed
        "ro_chs":[RO_CH], # --Fixed
        "reps":1, # --Fixed
        "relax_delay":1.0, # --us
        "res_phase":0, # --degrees
        "pulse_style": "const", # --Fixed
        
        "length":100, # [Clock ticks]
        # Try varying length from 10-100 clock ticks
    
        "readout_length":1000, # [Clock ticks]
        # Try varying readout_length from 50-1000 clock ticks

        "pulse_gain": 30000, # [DAC units]
        # Try varying pulse_gain from 500 to 30000 DAC units

        "pulse_freq": 200, # In this program the signal is up and downconverted digitally so you won't see any frequency
        # components in the I/Q traces below. But since the signal gain depends on frequency, 
        # if you lower pulse_freq you will see an increased gain.

        "adc_trig_offset": 100, # [Clock ticks]
        # Try varying adc_trig_offset from 100 to 220 clock ticks

        "soft_avgs":100
        # Try varying soft_avgs from 1 to 200 averages

       }

###################
# Try it yourself !
###################

prog = SingleTone(soccfg, config)
avgi, avgq = prog.acquire(soc)

  0%|          | 0/100 [00:00<?, ?it/s]

In [88]:
soc.reset_gens()

Continous Multi-tone Sine Wave

In [8]:
import numpy as np
from qick import AveragerProgram

class ManyToneWaveform(AveragerProgram):
    def initialize(self):
        cfg = self.cfg   
        res_ch = cfg["res_ch"]

        # Set the Nyquist zone
        self.declare_gen(ch=res_ch, nqz=1)
        
        # Configure the readouts
        for ch in cfg["ro_chs"]:
            self.declare_readout(ch=ch, length=cfg["readout_length"],
                                 freq=cfg["pulse_freq"], gen_ch=res_ch)

        # Setup standard channel registers (NCO center frequency, phase, and gain)
        freq = self.freq2reg(cfg["pulse_freq"], gen_ch=res_ch, ro_ch=cfg["ro_chs"][0])
        phase = self.deg2reg(cfg["res_phase"], gen_ch=res_ch)
        gain = cfg["pulse_gain"]
        self.default_pulse_registers(ch=res_ch, freq=freq, phase=phase, gain=gain)

        # Multi-tone synthesis from config list
        samps_per_clk = self.soccfg['gens'][res_ch]['samps_per_clk']
        gen_fs = self.soccfg['gens'][res_ch]['fs']
        print(gen_fs)
        fabric_freq_mhz = gen_fs / samps_per_clk
        
        # Pull the list of target offsets from the config
        tone_offsets = cfg["tone_offsets"]  
        num_tones = len(tone_offsets)
        
        # Maximize the pulse length to compress the frequency step size down
        max_allowed_cycles = 65536 // samps_per_clk
        cfg["pulse_length"] = int(max_allowed_cycles - (max_allowed_cycles % 16))
        
        # Establish the exact resolution grid step size (in MHz)
        total_duration_us = cfg["pulse_length"] / fabric_freq_mhz
        resolution_limit_mhz = 1.0 / total_duration_us
        
        print(f"--- Optimized Frequency Grid Step: {resolution_limit_mhz * 1e3:.3f} kHz ---")
        print(f"Snapping {num_tones} tones to the hardware grid:")

        # Initialize raw arrays for I and Q channels
        num_samples = cfg["pulse_length"] * samps_per_clk
        t = np.arange(num_samples)
        idata_raw = np.zeros(num_samples, dtype=np.float64)
        qdata_raw = np.zeros(num_samples, dtype=np.float64)
        
        # Loop through your arbitrary list of tones
        for idx, target_offset in enumerate(tone_offsets):
            # Snap the requested frequency to the closest physical loop bin
            snapped_offset = np.round(target_offset / resolution_limit_mhz) * resolution_limit_mhz
            print(f"  Tone {idx+1}: {target_offset} MHz -> {snapped_offset:.4f} MHz")
            
            # Normalize the frequency to the DAC sampling rate
            f_norm = snapped_offset / gen_fs
            
            # Programmatic Schroeder Phase calculation for N arbitrary tones
            phi = (np.pi * (idx**2)) / num_tones
            
            # Accumulate Single-Sideband vector components (Cos for I, Sin for Q)
            idata_raw += np.cos(2 * np.pi * f_norm * t + phi)
            qdata_raw += np.sin(2 * np.pi * f_norm * t + phi)
            
        # Normalize the aggregated complex waveform to prevent digital clipping
        complex_envelope = idata_raw + 1j * qdata_raw
        max_peak = np.max(np.abs(complex_envelope))
        
        max_dac_val = 30000 
        idata = ((idata_raw / max_peak) * max_dac_val).astype(np.int16)
        qdata = ((qdata_raw / max_peak) * max_dac_val).astype(np.int16)
        
        # Add the custom multi-tone waveform to the pulse library
        self.add_pulse(ch=res_ch, name="multi_tone", idata=idata, qdata=qdata)
        
        # Route the pulse to the channel
        self.set_pulse_registers(ch=res_ch, style="arb", waveform="multi_tone", mode="periodic")
        
        self.synci(200)
    
    def body(self):
        # Fire the pulse, trigger ADCs, and measure
        self.measure(pulse_ch=self.cfg["res_ch"], 
                     adcs=self.cfg["ro_chs"],
                     pins=[0], 
                     adc_trig_offset=self.cfg["adc_trig_offset"],
                     wait=True,
                     syncdelay=self.us2cycles(self.cfg["relax_delay"]))

In [9]:
config = {
    "res_ch": GEN_CH,
    "ro_chs": [RO_CH],
    "reps": 1,
    "relax_delay": 1.0,
    "res_phase": 0,
    
    # Pulse parameters
    "pulse_style": "arb", 
    "pulse_gain": 30000,   # [DAC units]
    "pulse_freq": 200,    # [MHz] Center NCO frequency
    
    # Pass an arbitrary list of tone offsets (in MHz from pulse_freq)
    "tone_offsets": [-15.2, -7.0, 3.5, 8.1, 14.0, 19.3], 
    
    "readout_length": 1000,
    "adc_trig_offset": 100,
    "soft_avgs": 100
}

prog = ManyToneWaveform(soccfg, config)
avgi, avgq = prog.acquire(soc)

6144.0
--- Optimized Frequency Grid Step: 93.750 kHz ---
Snapping 6 tones to the hardware grid:
  Tone 1: -15.2 MHz -> -15.1875 MHz
  Tone 2: -7.0 MHz -> -7.0312 MHz
  Tone 3: 3.5 MHz -> 3.4688 MHz
  Tone 4: 8.1 MHz -> 8.0625 MHz
  Tone 5: 14.0 MHz -> 13.9688 MHz
  Tone 6: 19.3 MHz -> 19.3125 MHz


  0%|          | 0/100 [00:00<?, ?it/s]

In [47]:
soc.reset_gens()

Without Frequency Grid

In [ ]:
import numpy as np
from qick import AveragerProgram

class ManyUnsnappedTones(AveragerProgram):
    def initialize(self):
        cfg = self.cfg   
        res_ch = cfg["res_ch"]

        # Set the Nyquist zone
        self.declare_gen(ch=res_ch, nqz=1)
        
        # Configure the readouts
        for ch in cfg["ro_chs"]:
            self.declare_readout(ch=ch, length=cfg["readout_length"],
                                 freq=cfg["pulse_freq"], gen_ch=res_ch)

        # Setup standard channel registers (NCO center frequency, phase, and gain)
        freq = self.freq2reg(cfg["pulse_freq"], gen_ch=res_ch, ro_ch=cfg["ro_chs"][0])
        phase = self.deg2reg(cfg["res_phase"], gen_ch=res_ch)
        gain = cfg["pulse_gain"]
        self.default_pulse_registers(ch=res_ch, freq=freq, phase=phase, gain=gain)

        # Multi-tone synthesis from configs
        samps_per_clk = self.soccfg['gens'][res_ch]['samps_per_clk']
        gen_fs = self.soccfg['gens'][res_ch]['fs']
        
        # Pull the list of target offsets from the config
        tone_offsets = cfg["tone_offsets"]  
        num_tones = len(tone_offsets)

        # Initialize raw arrays for I and Q channels using the user-defined pulse_length
        num_samples = cfg["pulse_length"] * samps_per_clk
        t = np.arange(num_samples)
        idata_raw = np.zeros(num_samples, dtype=np.float64)
        qdata_raw = np.zeros(num_samples, dtype=np.float64)
        
        # Loop through your arbitrary list of tones
        for idx, target_offset in enumerate(tone_offsets):
            print(f"  Tone {idx+1}: {target_offset} MHz (Direct)")
            
            # Normalize the target frequency directly to the DAC sampling rate
            f_norm = target_offset / gen_fs
            
            # Programmatic Schroeder Phase calculation for N arbitrary tones
            phi = (np.pi * (idx**2)) / num_tones
            
            # Accumulate Single-Sideband vector components (Cos for I, Sin for Q)
            idata_raw += np.cos(2 * np.pi * f_norm * t + phi)
            qdata_raw += np.sin(2 * np.pi * f_norm * t + phi)
            
        # Normalize the aggregated complex waveform to prevent digital clipping
        complex_envelope = idata_raw + 1j * qdata_raw
        max_peak = np.max(np.abs(complex_envelope))
        
        max_dac_val = 30000 
        idata = ((idata_raw / max_peak) * max_dac_val).astype(np.int16)
        qdata = ((qdata_raw / max_peak) * max_dac_val).astype(np.int16)
        
        # Add the custom multi-tone waveform to the pulse library
        self.add_pulse(ch=res_ch, name="multi_tone", idata=idata, qdata=qdata)
        
        # 5. Route the pulse to the channel
        self.set_pulse_registers(ch=res_ch, style="arb", waveform="multi_tone", mode="periodic")
        
        self.synci(200)
    
    def body(self):
        # Fire the pulse, trigger ADCs, and measure
        self.measure(pulse_ch=self.cfg["res_ch"], 
                     adcs=self.cfg["ro_chs"],
                     pins=[0], 
                     adc_trig_offset=self.cfg["adc_trig_offset"],
                     wait=True,
                     syncdelay=self.us2cycles(self.cfg["relax_delay"]))

In [ ]:
config = {
    "res_ch": GEN_CH,
    "ro_chs": [RO_CH],
    "reps": 1,
    "relax_delay": 1.0,
    "res_phase": 0,
    
    # Pulse parameters
    "pulse_style": "arb", 
    "pulse_length": 200,
    "pulse_gain": 30000,   # [DAC units]
    "pulse_freq": 200,    # [MHz] Center NCO frequency
    
    # Pass an arbitrary list of tone offsets (in MHz from pulse_freq)
    "tone_offsets": [-15.2, -7.0, 3.5, 8.1, 14.0, 19.3], 
    
    "readout_length": 1000,
    "adc_trig_offset": 100,
    "soft_avgs": 100
}

prog = ManyUnsnappedTones(soccfg, config)
avgi, avgq = prog.acquire(soc)